# Analysis of Emergency Obstetric Care (EmOC) in Abuja
> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../kano/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate isochrones on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd


import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [3]:
# Set paths to access Kano data
# Define directories
data_inputs = '../scripts/Abuja/data-inputs/'
data_temp = '../scripts/Abuja/data-temp/'
model_outputs = '../abuja/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined with the assistance of local experts, based on data obtained from the [datasets of health facilities](https://doi.org/10.6084/m9.figshare.22689667.v2).

In [11]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities_abuja_emoc.geojson')
healthcare_facilities_validated

,orig_order,state,lga,ward,urban_conurb,facility_code,ontime_code,facility_name,owner,specific_owner,...,verif_onground,alt_name_fac,latitude,longitude,operation_status,registration_status,license_status,local_validation,hcf_id,geometry
0,693,3,Abuja Municipal Area Council,Wuse,3,37/06/1/1/1/0042,100302001,House Clinic (Asokoro),1,1,...,True,State House Clinic/The State House Medical Centre,9.062250,7.518720,Operational,Unknown,Unknown,Public Comprehensive EmOC,0,POINT (7.51872 9.06225)
1,694,3,Abuja Municipal Area Council,Wuse,3,37/06/1/1/2/0114,100302002,Dr. Hassan's Clinic and Diagnostic Centre,2,2,...,False,None,9.094030,7.495550,Operational,Unknown,Unknown,Private Comprehensive EmOC,1,POINT (7.49555 9.09403)
2,695,3,Abuja Municipal Area Council,Wuse,3,37/06/1/2/1/0006,100302003,Maitama General Hospital,1,1,...,False,None,9.086310,7.481390,Operational,Unknown,Unknown,Public Comprehensive EmOC,2,POINT (7.48139 9.08631)
3,696,3,Abuja Municipal Area Council,Wuse,3,37/06/1/1/2/0323,100302004,Wuse Clinic and Maternity,2,2,...,False,None,9.069440,7.478980,Operational,Unknown,Unknown,Private Comprehensive EmOC,3,POINT (7.47898 9.06944)
4,697,3,Abuja Municipal Area Council,Wuse,3,37/06/1/2/2/0002,100302005,Chivar Clinic and Urology,2,2,...,False,Chivar Specialist hospital and Centre,9.061960,7.474930,Operational,Unknown,Unknown,Private Comprehensive EmOC,4,POINT (7.47493 9.06196)
5,698,3,Abuja Municipal Area Council,Wuse,3,37/06/1/3/2/0001,100302006,King's Care Hospital and Maternity (Wuse),2,2,...,False,None,9.065770,7.473970,Operational,Unknown,Unknown,Private Comprehensive EmOC,5,POINT (7.47397 9.06577)
6,699,3,Abuja Municipal Area Council,Wuse,3,37/06/1/2/2/0038,100302007,Sami Wadata Clinic (Wuse),2,2,...,True,None,9.086650,7.470420,Operational,Registered,Licensed,Private Comprehensive EmOC,6,POINT (7.47042 9.08665)
7,700,3,Abuja Municipal Area Council,Wuse,3,37/06/1/2/1/0013,100302008,Wuse General Hospital,1,1,...,False,None,9.062940,7.468900,Operational,Not Applicable,Not Applicable,Public Comprehensive EmOC,7,POINT (7.4689 9.06294)
8,701,3,Abuja Municipal Area Council,Wuse,3,37/06/1/1/2/0250,100302009,Queens Clinic and Maternity,2,2,...,False,None,9.069840,7.462650,Operational,Unknown,Unknown,Private Comprehensive EmOC,8,POINT (7.46265 9.06984)
9,702,3,Abuja Municipal Area Council,City Centre,3,37/06/1/3/1/0002,100302010,National Hospital (Abuja),1,1,...,False,None,9.039060,7.461550,Operational,Registered,Licensed,Public Comprehensive EmOC,9,POINT (7.46155 9.03906)


In [12]:
healthcare_facilities_validated['hcf_id'] = range(len(healthcare_facilities_validated))
healthcare_facilities_validated = healthcare_facilities_validated[['hcf_id', 'facility_name', 'longitude', 'latitude', 'local_validation', 'geometry']]
healthcare_facilities_validated

,hcf_id,facility_name,longitude,latitude,local_validation,geometry
0,0,House Clinic (Asokoro),7.518720,9.062250,Public Comprehensive EmOC,POINT (7.51872 9.06225)
1,1,Dr. Hassan's Clinic and Diagnostic Centre,7.495550,9.094030,Private Comprehensive EmOC,POINT (7.49555 9.09403)
2,2,Maitama General Hospital,7.481390,9.086310,Public Comprehensive EmOC,POINT (7.48139 9.08631)
3,3,Wuse Clinic and Maternity,7.478980,9.069440,Private Comprehensive EmOC,POINT (7.47898 9.06944)
4,4,Chivar Clinic and Urology,7.474930,9.061960,Private Comprehensive EmOC,POINT (7.47493 9.06196)
5,5,King's Care Hospital and Maternity (Wuse),7.473970,9.065770,Private Comprehensive EmOC,POINT (7.47397 9.06577)
6,6,Sami Wadata Clinic (Wuse),7.470420,9.086650,Private Comprehensive EmOC,POINT (7.47042 9.08665)
7,7,Wuse General Hospital,7.468900,9.062940,Public Comprehensive EmOC,POINT (7.4689 9.06294)
8,8,Queens Clinic and Maternity,7.462650,9.069840,Private Comprehensive EmOC,POINT (7.46265 9.06984)
9,9,National Hospital (Abuja),7.461550,9.039060,Public Comprehensive EmOC,POINT (7.46155 9.03906)


In [13]:
healthcare_facilities_validated.to_file(data_inputs + 'healthcare_facilities_abuja_emoc.geojson', driver='GeoJSON')

### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [3]:
study_area = gpd.read_file(data_inputs + 'grid-boundary-abuja.gpkg')
raster_path = data_inputs + 'nga_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [4]:
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_32136/2284915905.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


In [5]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [6]:
with rasterio.open(data_inputs + 'abuja_nga_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

### Adding population data at 1km grid to 100m grid

In [3]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

epsg = 'EPSG:32632'

In [4]:
# Preparing grid
grid_file = data_inputs + 'grid-boundary-abuja.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'geometry','latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,"POLYGON ((320437.685 1031975.076, 320439.266 1...",9.332425,9.332019,9.332831,7.364534,7.364025,7.365042
1,1,"POLYGON ((320548.2 1031974.564, 320549.781 103...",9.332425,9.332019,9.332831,7.365540,7.365031,7.366048
2,2,"POLYGON ((320658.716 1031974.053, 320660.296 1...",9.332425,9.332019,9.332831,7.366546,7.366037,7.367054
3,3,"POLYGON ((320769.231 1031973.543, 320770.811 1...",9.332425,9.332019,9.332831,7.367552,7.367043,7.368060
4,4,"POLYGON ((320879.746 1031973.032, 320881.326 1...",9.332425,9.332019,9.332831,7.368558,7.368049,7.369066
...,...,...,...,...,...,...,...,...
142495,142495,"POLYGON ((359959.239 983370.132, 359960.712 98...",8.894373,8.893967,8.894779,7.725878,7.725370,7.726386
142496,142496,"POLYGON ((360069.788 983369.752, 360071.261 98...",8.894373,8.893967,8.894779,7.726883,7.726375,7.727391
142497,142497,"POLYGON ((360180.338 983369.372, 360181.811 98...",8.894373,8.893967,8.894779,7.727888,7.727380,7.728396
142498,142498,"POLYGON ((360290.887 983368.993, 360292.36 983...",8.894373,8.893967,8.894779,7.728894,7.728386,7.729402


Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.

In [5]:
# Count buildings per grid cell

# Load Google building footprints
building_file = data_inputs + 'abuja_GOB.parquet'
buildings = gpd.read_parquet(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

for df in [grid, buildings]:
    cols_to_remove = [c for c in df.columns if c.startswith('index_') or c.endswith('_left') or c.endswith('_right')]
    if cols_to_remove:
        df.drop(columns=cols_to_remove, inplace=True)

# Join buildings to grid using centroid
grid_buildings = grid.sjoin(
    buildings.set_geometry('centroid').drop(columns='geometry'),
    how='inner',
    predicate='intersects'
)

# Count buildings per grid cell
building_counts = grid_buildings.groupby('grid_id').size().rename('bcount')

# Add building count to grid
grid = grid.merge(building_counts, on='grid_id', how='left')
grid['bcount'] = grid['bcount'].fillna(0)   # assign 0 to empty cells
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max,bcount
0,0,"POLYGON ((320437.685 1031975.076, 320439.266 1...",9.332425,9.332019,9.332831,7.364534,7.364025,7.365042,0.0
1,1,"POLYGON ((320548.2 1031974.564, 320549.781 103...",9.332425,9.332019,9.332831,7.365540,7.365031,7.366048,0.0
2,2,"POLYGON ((320658.716 1031974.053, 320660.296 1...",9.332425,9.332019,9.332831,7.366546,7.366037,7.367054,0.0
3,3,"POLYGON ((320769.231 1031973.543, 320770.811 1...",9.332425,9.332019,9.332831,7.367552,7.367043,7.368060,0.0
4,4,"POLYGON ((320879.746 1031973.032, 320881.326 1...",9.332425,9.332019,9.332831,7.368558,7.368049,7.369066,0.0
...,...,...,...,...,...,...,...,...,...
142495,142495,"POLYGON ((359959.239 983370.132, 359960.712 98...",8.894373,8.893967,8.894779,7.725878,7.725370,7.726386,0.0
142496,142496,"POLYGON ((360069.788 983369.752, 360071.261 98...",8.894373,8.893967,8.894779,7.726883,7.726375,7.727391,1.0
142497,142497,"POLYGON ((360180.338 983369.372, 360181.811 98...",8.894373,8.893967,8.894779,7.727888,7.727380,7.728396,1.0
142498,142498,"POLYGON ((360290.887 983368.993, 360292.36 983...",8.894373,8.893967,8.894779,7.728894,7.728386,7.729402,0.0


The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [6]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Load coarse population raster
pop_file = data_path / 'abuja_nga_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Convert raster to vector population grid
pop_grid = raster2vector(pop_raster, transform, crs)
pop_grid = pop_grid.to_crs(epsg)
pop_grid['pop_grid_id'] = range(len(pop_grid))
pop_grid.to_csv(data_path / 'pop_grid_id.csv')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

Index(['grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max', 'longitude',
       'lon_min', 'lon_max', 'bcount', 'centroid', 'index_right',
       'pop_grid_pop', 'pop_grid_id'],
      dtype='object')


,grid_id,bcount,pop_grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,0.0,32,"POLYGON ((320437.685 1031975.076, 320439.266 1...",9.332425,9.332019,9.332831,7.364534,7.364025,7.365042
1,1,0.0,32,"POLYGON ((320548.2 1031974.564, 320549.781 103...",9.332425,9.332019,9.332831,7.365540,7.365031,7.366048
2,2,0.0,32,"POLYGON ((320658.716 1031974.053, 320660.296 1...",9.332425,9.332019,9.332831,7.366546,7.366037,7.367054
3,3,0.0,32,"POLYGON ((320769.231 1031973.543, 320770.811 1...",9.332425,9.332019,9.332831,7.367552,7.367043,7.368060
4,4,0.0,32,"POLYGON ((320879.746 1031973.032, 320881.326 1...",9.332425,9.332019,9.332831,7.368558,7.368049,7.369066


In [7]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,geometry_y,pop
0,0,0.0,32,"POLYGON ((320437.685 1031975.076, 320439.266 1...",9.332425,9.332019,9.332831,7.364534,7.364025,7.365042,0.0,NaN,NaN,"POLYGON ((319933.343 1032537.621, 320848.785 1...",NaN
1,1,0.0,32,"POLYGON ((320548.2 1031974.564, 320549.781 103...",9.332425,9.332019,9.332831,7.365540,7.365031,7.366048,0.0,NaN,NaN,"POLYGON ((319933.343 1032537.621, 320848.785 1...",NaN
2,2,0.0,32,"POLYGON ((320658.716 1031974.053, 320660.296 1...",9.332425,9.332019,9.332831,7.366546,7.366037,7.367054,0.0,NaN,NaN,"POLYGON ((319933.343 1032537.621, 320848.785 1...",NaN
3,3,0.0,32,"POLYGON ((320769.231 1031973.543, 320770.811 1...",9.332425,9.332019,9.332831,7.367552,7.367043,7.368060,0.0,NaN,NaN,"POLYGON ((319933.343 1032537.621, 320848.785 1...",NaN
4,4,0.0,32,"POLYGON ((320879.746 1031973.032, 320881.326 1...",9.332425,9.332019,9.332831,7.368558,7.368049,7.369066,0.0,NaN,NaN,"POLYGON ((319933.343 1032537.621, 320848.785 1...",NaN


In [8]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])

# Keep all grid cells including those with zero population
grid["pop"] = (
    grid["pop"]
    .fillna(0)
)

In [9]:
grid

,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop
0,0,0.0,32,"POLYGON ((320437.685 1031975.076, 320439.266 1...",9.332425,9.332019,9.332831,7.364534,7.364025,7.365042,0.0,NaN,NaN,0.0
1,1,0.0,32,"POLYGON ((320548.2 1031974.564, 320549.781 103...",9.332425,9.332019,9.332831,7.365540,7.365031,7.366048,0.0,NaN,NaN,0.0
2,2,0.0,32,"POLYGON ((320658.716 1031974.053, 320660.296 1...",9.332425,9.332019,9.332831,7.366546,7.366037,7.367054,0.0,NaN,NaN,0.0
3,3,0.0,32,"POLYGON ((320769.231 1031973.543, 320770.811 1...",9.332425,9.332019,9.332831,7.367552,7.367043,7.368060,0.0,NaN,NaN,0.0
4,4,0.0,32,"POLYGON ((320879.746 1031973.032, 320881.326 1...",9.332425,9.332019,9.332831,7.368558,7.368049,7.369066,0.0,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142495,142495,0.0,4156,"POLYGON ((359959.239 983370.132, 359960.712 98...",8.894373,8.893967,8.894779,7.725878,7.725370,7.726386,25.0,0.000,NaN,0.0
142496,142496,1.0,4156,"POLYGON ((360069.788 983369.752, 360071.261 98...",8.894373,8.893967,8.894779,7.726883,7.726375,7.727391,25.0,0.040,NaN,0.0
142497,142497,1.0,4157,"POLYGON ((360180.338 983369.372, 360181.811 98...",8.894373,8.893967,8.894779,7.727888,7.727380,7.728396,8.0,0.125,NaN,0.0
142498,142498,0.0,4157,"POLYGON ((360290.887 983368.993, 360292.36 983...",8.894373,8.893967,8.894779,7.728894,7.728386,7.729402,8.0,0.000,NaN,0.0


In [10]:
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-abuja.gpkg', driver='GPKG')

In [11]:
# Preparing gird centroids with population attribute for accessibility analysis
grid = gpd.read_file(data_temp + "pop-grid-abuja.gpkg")
grid_ll = grid.to_crs(epsg=4326)

grid_ll["geometry"] = grid_ll.geometry.centroid

grid_ll["latitude"] = grid_ll["geometry"].y
grid_ll["longitude"] = grid_ll["geometry"].x

grid_centroids = grid_ll[["grid_id", "latitude", "longitude", "geometry", "pop"]].copy()
grid_centroids = grid_centroids.set_geometry("geometry")
grid_centroids.set_crs("EPSG:4326", inplace=True)

grid_centroids.reset_index(drop=True, inplace=True)

grid_centroids.to_file(data_temp + "grid_centroids.gpkg", driver="GPKG")

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_32228/1285117187.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid_ll["geometry"] = grid_ll.geometry.centroid


In [12]:
grid_centroids

,grid_id,latitude,longitude,geometry,pop
0,0,9.332425,7.364534,POINT (7.36453 9.33242),0.0
1,1,9.332425,7.365540,POINT (7.36554 9.33242),0.0
2,2,9.332425,7.366546,POINT (7.36655 9.33242),0.0
3,3,9.332425,7.367552,POINT (7.36755 9.33242),0.0
4,4,9.332425,7.368558,POINT (7.36856 9.33242),0.0
...,...,...,...,...,...
142495,142495,8.894373,7.725878,POINT (7.72588 8.89437),0.0
142496,142496,8.894373,7.726883,POINT (7.72688 8.89437),0.0
142497,142497,8.894373,7.727888,POINT (7.72789 8.89437),0.0
142498,142498,8.894373,7.728894,POINT (7.72889 8.89437),0.0


## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [ ]:
origin_gdf = gpd.read_file(data_temp + "grid_centroids.gpkg")
origin_name_column = 'grid_id'
destination_gdf = gpd.read_file(data_inputs + 'healthcare_facilities_abuja_emoc.geojson').dropna(subset=['geometry'])
destination_name_column = 'hcf_id'

In [4]:
# Extract coordinates
origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))
locations = origins + destinations

In [5]:
# Indices
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

In [7]:
# Prepare API request
body = {
    'locations': locations,
    'destinations': destinations_index,
    'sources': origins_index,
    'metrics': ['distance', 'duration']
}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

# Make request
response = requests.post(
    'https://api.openrouteservice.org/v2/matrix/driving-car',
    json=body,
    headers=headers
)

In [8]:
# Parse response
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

In [ ]:
# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [4]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-abuja.gpkg')
centroids_df

,grid_id,bcount,pop_grid_id,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,0,0.0,32,9.332425,9.332019,9.332831,7.364534,7.364025,7.365042,0.0,NaN,NaN,0.0,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7...."
1,1,0.0,32,9.332425,9.332019,9.332831,7.365540,7.365031,7.366048,0.0,NaN,NaN,0.0,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7...."
2,2,0.0,32,9.332425,9.332019,9.332831,7.366546,7.366037,7.367054,0.0,NaN,NaN,0.0,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7...."
3,3,0.0,32,9.332425,9.332019,9.332831,7.367552,7.367043,7.368060,0.0,NaN,NaN,0.0,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7...."
4,4,0.0,32,9.332425,9.332019,9.332831,7.368558,7.368049,7.369066,0.0,NaN,NaN,0.0,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142495,142495,0.0,4156,8.894373,8.893967,8.894779,7.725878,7.725370,7.726386,25.0,0.000,NaN,0.0,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7...."
142496,142496,1.0,4156,8.894373,8.893967,8.894779,7.726883,7.726375,7.727391,25.0,0.040,NaN,0.0,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7...."
142497,142497,1.0,4157,8.894373,8.893967,8.894779,7.727888,7.727380,7.728396,8.0,0.125,NaN,0.0,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7..."
142498,142498,0.0,4157,8.894373,8.893967,8.894779,7.728894,7.728386,7.729402,8.0,0.000,NaN,0.0,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7..."


In [5]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'abuja_access.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,0,0.0,2827.42,52.31
1,0,1.0,2820.48,52.25
2,0,2.0,2810.50,52.17
3,0,3.0,2803.42,52.11
4,0,4.0,2800.56,52.08
...,...,...,...,...
3419995,23,142495.0,4298.73,97.21
3419996,23,142496.0,4302.10,97.22
3419997,23,142497.0,4275.63,97.25
3419998,23,142498.0,4278.42,97.32


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [6]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,0,0.0,2827.42,52.31
1,0,1.0,2820.48,52.25
2,0,2.0,2810.50,52.17
3,0,3.0,2803.42,52.11
4,0,4.0,2800.56,52.08
...,...,...,...,...
3419995,23,142495.0,4298.73,97.21
3419996,23,142496.0,4302.10,97.22
3419997,23,142497.0,4275.63,97.25
3419998,23,142498.0,4278.42,97.32


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [7]:
pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
                     left_on='destination_id', right_on='grid_id', how='left')
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,0,0.0,2827.42,52.31,0,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,NaN,0.0,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7...."
1,0,1.0,2820.48,52.25,1,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,NaN,0.0,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7...."
2,0,2.0,2810.50,52.17,2,7.366546,9.332425,7.366037,9.332019,7.367054,9.332831,0.0,0.0,NaN,0.0,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7...."
3,0,3.0,2803.42,52.11,3,7.367552,9.332425,7.367043,9.332019,7.368060,9.332831,0.0,0.0,NaN,0.0,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7...."
4,0,4.0,2800.56,52.08,4,7.368558,9.332425,7.368049,9.332019,7.369066,9.332831,0.0,0.0,NaN,0.0,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3418723,23,142495.0,4298.73,97.21,142495,7.725878,8.894373,7.725370,8.893967,7.726386,8.894779,0.0,25.0,NaN,0.0,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7...."
3418724,23,142496.0,4302.10,97.22,142496,7.726883,8.894373,7.726375,8.893967,7.727391,8.894779,1.0,25.0,NaN,0.0,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7...."
3418725,23,142497.0,4275.63,97.25,142497,7.727888,8.894373,7.727380,8.893967,7.728396,8.894779,1.0,8.0,NaN,0.0,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7..."
3418726,23,142498.0,4278.42,97.32,142498,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,8.0,NaN,0.0,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7..."


In [8]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "origin_id": "hcf_id",
    "pop": "population"
})
columns_to_keep = ["grid_id", "hcf_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

In [9]:
pop_centroids_hcf

,grid_id,hcf_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km
0,0,0,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2827.42,52.31
1,1,0,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2820.48,52.25
2,2,0,7.366546,9.332425,7.366037,9.332019,7.367054,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7....",2810.50,52.17
3,3,0,7.367552,9.332425,7.367043,9.332019,7.368060,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7....",2803.42,52.11
4,4,0,7.368558,9.332425,7.368049,9.332019,7.369066,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7....",2800.56,52.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3418723,142495,23,7.725878,8.894373,7.725370,8.893967,7.726386,8.894779,0.0,0.0,25.0,NaN,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7....",4298.73,97.21
3418724,142496,23,7.726883,8.894373,7.726375,8.893967,7.727391,8.894779,0.0,1.0,25.0,NaN,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7....",4302.10,97.22
3418725,142497,23,7.727888,8.894373,7.727380,8.893967,7.728396,8.894779,0.0,1.0,8.0,NaN,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7...",4275.63,97.25
3418726,142498,23,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",4278.42,97.32


Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [14]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id', 'facility_name', 'local_validation']], 
                     left_on='hcf_id', right_on='hcf_id', how='left') # left on is from od matrix, right on is from healthcare facilities

In [15]:
distances_duration_matrix

,grid_id,hcf_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,facility_name,local_validation
0,0,0,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2827.42,52.31,House Clinic (Asokoro),Public Comprehensive EmOC
1,1,0,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2820.48,52.25,House Clinic (Asokoro),Public Comprehensive EmOC
2,2,0,7.366546,9.332425,7.366037,9.332019,7.367054,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7....",2810.50,52.17,House Clinic (Asokoro),Public Comprehensive EmOC
3,3,0,7.367552,9.332425,7.367043,9.332019,7.368060,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7....",2803.42,52.11,House Clinic (Asokoro),Public Comprehensive EmOC
4,4,0,7.368558,9.332425,7.368049,9.332019,7.369066,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7....",2800.56,52.08,House Clinic (Asokoro),Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3418723,142495,23,7.725878,8.894373,7.725370,8.893967,7.726386,8.894779,0.0,0.0,25.0,NaN,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7....",4298.73,97.21,Federal Staff Hospital (Abuja),Public Comprehensive EmOC
3418724,142496,23,7.726883,8.894373,7.726375,8.893967,7.727391,8.894779,0.0,1.0,25.0,NaN,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7....",4302.10,97.22,Federal Staff Hospital (Abuja),Public Comprehensive EmOC
3418725,142497,23,7.727888,8.894373,7.727380,8.893967,7.728396,8.894779,0.0,1.0,8.0,NaN,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7...",4275.63,97.25,Federal Staff Hospital (Abuja),Public Comprehensive EmOC
3418726,142498,23,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",4278.42,97.32,Federal Staff Hospital (Abuja),Public Comprehensive EmOC


In [16]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "longitude": "dest_lon",
    "latitude": "dest_lat"
})

In [17]:
category_counts = healthcare_facilities_validated['local_validation'].value_counts()
print(category_counts)

local_validation
Public Comprehensive EmOC     13
Private Comprehensive EmOC    11
Name: count, dtype: int64


In [18]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC']

In [19]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['local_validation'].str.contains(values, na=False)]
    for key, values in zip(selected_categories, selected_categories)
}

public_CEmOC = subsets["Public Comprehensive EmOC"]
private_CEmOC = subsets["Private Comprehensive EmOC"]

In [20]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)

In [21]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_74427/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)
/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_74427/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsm

In [22]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([  
    public_CEmOC_closest_3, private_CEmOC_closest_3
])
distances_duration_matrix

,grid_id,hcf_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,facility_name,local_validation
0,0,17,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2412.47,43.20,General Hospital Maitama (Mdh),Public Comprehensive EmOC
1,0,18,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2415.92,43.09,Abuja Police Hospital,Public Comprehensive EmOC
2,0,2,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2435.20,43.66,Maitama General Hospital,Public Comprehensive EmOC
3,1,17,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2419.41,43.25,General Hospital Maitama (Mdh),Public Comprehensive EmOC
4,1,18,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2422.86,43.15,Abuja Police Hospital,Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427336,142498,3,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",1800.17,40.68,Wuse Clinic and Maternity,Private Comprehensive EmOC
427337,142498,4,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",1807.02,41.91,Chivar Clinic and Urology,Private Comprehensive EmOC
427338,142499,5,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.7304 8.89397, 7.73041 8.89478, 7.7...",2080.66,48.74,King's Care Hospital and Maternity (Wuse),Private Comprehensive EmOC
427339,142499,3,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.7304 8.89397, 7.73041 8.89478, 7.7...",2112.99,47.93,Wuse Clinic and Maternity,Private Comprehensive EmOC


In [23]:
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")

gpkg_path = data_temp + "distances_duration_3_closet_Emoc.gpkg"
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG", mode="w")


In [24]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [25]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [26]:
print(origin_dest.head())

   grid_id  hcf_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        0      17    7.364534    9.332425        7.364025        9.332019   
1        0      18    7.364534    9.332425        7.364025        9.332019   
2        0       2    7.364534    9.332425        7.364025        9.332019   
3        1      17    7.365540    9.332425        7.365031        9.332019   
4        1      18    7.365540    9.332425        7.365031        9.332019   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0        7.365042        9.332831         0.0     0.0              0.0   
1        7.365042        9.332831         0.0     0.0              0.0   
2        7.365042        9.332831         0.0     0.0              0.0   
3        7.366048        9.332831         0.0     0.0              0.0   
4        7.366048        9.332831         0.0     0.0              0.0   

   pop_grid_pop                                           geometry  \
0           NaN 

In [27]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [28]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [29]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [30]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [31]:
origin_dest_acc

,grid_id,hcf_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,facility_name,local_validation,Weight,Pop_W
0,0,17,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2412.47,43.20,General Hospital Maitama (Mdh),Public Comprehensive EmOC,0.0,0.0
1,0,18,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2415.92,43.09,Abuja Police Hospital,Public Comprehensive EmOC,0.0,0.0
2,0,2,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2435.20,43.66,Maitama General Hospital,Public Comprehensive EmOC,0.0,0.0
3,1,17,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2419.41,43.25,General Hospital Maitama (Mdh),Public Comprehensive EmOC,0.0,0.0
4,1,18,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2422.86,43.15,Abuja Police Hospital,Public Comprehensive EmOC,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427336,142498,3,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",1800.17,40.68,Wuse Clinic and Maternity,Private Comprehensive EmOC,0.0,0.0
427337,142498,4,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",1807.02,41.91,Chivar Clinic and Urology,Private Comprehensive EmOC,0.0,0.0
427338,142499,5,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.7304 8.89397, 7.73041 8.89478, 7.7...",2080.66,48.74,King's Care Hospital and Maternity (Wuse),Private Comprehensive EmOC,0.0,0.0
427339,142499,3,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.7304 8.89397, 7.73041 8.89478, 7.7...",2112.99,47.93,Wuse Clinic and Maternity,Private Comprehensive EmOC,0.0,0.0


In [32]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [33]:
origin_dest_sum

,hcf_id,Pop_W
0,0,6636.945942
1,1,2040.850412
2,2,13975.044019
3,3,12044.298414
4,4,10124.382471
5,5,16893.065648
6,6,5919.142440
7,7,12912.758434
8,8,6314.919795
9,9,4679.626881


In [34]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')

In [35]:
origin_dest_acc

,grid_id,hcf_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,facility_name,local_validation,Weight,Pop_W_x,Pop_W_y
0,0,17,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2412.47,43.20,General Hospital Maitama (Mdh),Public Comprehensive EmOC,0.0,0.0,9192.524083
1,0,18,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2415.92,43.09,Abuja Police Hospital,Public Comprehensive EmOC,0.0,0.0,9280.260737
2,0,2,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2435.20,43.66,Maitama General Hospital,Public Comprehensive EmOC,0.0,0.0,13975.044019
3,1,17,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2419.41,43.25,General Hospital Maitama (Mdh),Public Comprehensive EmOC,0.0,0.0,9192.524083
4,1,18,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,0.0,NaN,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2422.86,43.15,Abuja Police Hospital,Public Comprehensive EmOC,0.0,0.0,9280.260737
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854677,142498,3,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",1800.17,40.68,Wuse Clinic and Maternity,Private Comprehensive EmOC,0.0,0.0,12044.298414
854678,142498,4,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",1807.02,41.91,Chivar Clinic and Urology,Private Comprehensive EmOC,0.0,0.0,10124.382471
854679,142499,5,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.7304 8.89397, 7.73041 8.89478, 7.7...",2080.66,48.74,King's Care Hospital and Maternity (Wuse),Private Comprehensive EmOC,0.0,0.0,16893.065648
854680,142499,3,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,8.0,NaN,"POLYGON ((7.7304 8.89397, 7.73041 8.89478, 7.7...",2112.99,47.93,Wuse Clinic and Maternity,Private Comprehensive EmOC,0.0,0.0,12044.298414


In [36]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [37]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7
}

In [38]:
origin_dest_acc['supply'] = origin_dest_acc['local_validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_74427/4265979437.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [39]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [40]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [41]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [42]:
origin_dest_acc

,grid_id,hcf_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,...,facility_name,local_validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,0,17,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,...,General Hospital Maitama (Mdh),Public Comprehensive EmOC,0.0,0.0,9192.524083,1.0,0.000109,0.0,0.0,0.0
1,0,18,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,...,Abuja Police Hospital,Public Comprehensive EmOC,0.0,0.0,9280.260737,1.0,0.000108,0.0,0.0,0.0
2,0,2,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,0.0,...,Maitama General Hospital,Public Comprehensive EmOC,0.0,0.0,13975.044019,1.0,0.000072,0.0,0.0,0.0
3,1,17,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,...,General Hospital Maitama (Mdh),Public Comprehensive EmOC,0.0,0.0,9192.524083,1.0,0.000109,0.0,0.0,0.0
4,1,18,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,0.0,...,Abuja Police Hospital,Public Comprehensive EmOC,0.0,0.0,9280.260737,1.0,0.000108,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854677,142498,3,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,...,Wuse Clinic and Maternity,Private Comprehensive EmOC,0.0,0.0,12044.298414,0.7,0.000058,0.0,0.0,0.0
854678,142498,4,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,0.0,...,Chivar Clinic and Urology,Private Comprehensive EmOC,0.0,0.0,10124.382471,0.7,0.000069,0.0,0.0,0.0
854679,142499,5,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,...,King's Care Hospital and Maternity (Wuse),Private Comprehensive EmOC,0.0,0.0,16893.065648,0.7,0.000041,0.0,0.0,0.0
854680,142499,3,7.729899,8.894373,7.729391,8.893967,7.730407,8.894779,0.0,0.0,...,Wuse Clinic and Maternity,Private Comprehensive EmOC,0.0,0.0,12044.298414,0.7,0.000058,0.0,0.0,0.0


In [43]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [44]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [81]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [82]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [83]:
# Group by multiple columns and calculate the mean for numeric columns
# results_grid = results_grid.groupby(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard']).count().reset_index()
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])
type(results_grid)

geopandas.geodataframe.GeoDataFrame

In [84]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

In [85]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,0,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7...."
3,1,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7...."
6,2,7.366546,9.332425,7.366037,9.332019,7.367054,9.332831,0.0,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7...."
9,3,7.367552,9.332425,7.367043,9.332019,7.368060,9.332831,0.0,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7...."
12,4,7.368558,9.332425,7.368049,9.332019,7.369066,9.332831,0.0,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7...."
...,...,...,...,...,...,...,...,...,...
427326,142495,7.725878,8.894373,7.725370,8.893967,7.726386,8.894779,0.0,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7...."
427329,142496,7.726883,8.894373,7.726375,8.893967,7.727391,8.894779,0.0,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7...."
427332,142497,7.727888,8.894373,7.727380,8.893967,7.728396,8.894779,0.0,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7..."
427335,142498,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [86]:
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] >= 0, 'result'] = 2
results_grid.loc[results_grid['Accessibility_standard'] > 0.0000000001, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.0000087682, 'result'] = 0

In [87]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry,result
0,0,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2
3,1,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2
6,2,7.366546,9.332425,7.366037,9.332019,7.367054,9.332831,0.0,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7....",2
9,3,7.367552,9.332425,7.367043,9.332019,7.368060,9.332831,0.0,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7....",2
12,4,7.368558,9.332425,7.368049,9.332019,7.369066,9.332831,0.0,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7....",2
...,...,...,...,...,...,...,...,...,...,...
427326,142495,7.725878,8.894373,7.725370,8.893967,7.726386,8.894779,0.0,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7....",2
427329,142496,7.726883,8.894373,7.726375,8.893967,7.727391,8.894779,0.0,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7....",2
427332,142497,7.727888,8.894373,7.727380,8.893967,7.728396,8.894779,0.0,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7...",2
427335,142498,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",2


In [88]:
category_counts = results_grid['result'].value_counts()
print(category_counts)

result
1    48254
0    47483
2    46710
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [89]:
results_grid['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid.loc[(results_grid['Accessibility_standard'] > 0) & (results_grid['Accessibility_standard'] < 0.000000000001), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.0000000001) & (results_grid['Accessibility_standard'] < 0.0000000007), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.00000872) & (results_grid['Accessibility_standard'] < 0.0000088), 'focused'] = 1

In [90]:
results_grid = results_grid.loc[results_grid['result'] != -1]

In [91]:
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [92]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [93]:
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
0,0,7.364534,9.332425,7.364025,9.332019,7.365042,9.332831,0.0,"POLYGON ((7.36503 9.33202, 7.36504 9.33283, 7....",2,0
3,1,7.365540,9.332425,7.365031,9.332019,7.366048,9.332831,0.0,"POLYGON ((7.36604 9.33202, 7.36605 9.33283, 7....",2,0
6,2,7.366546,9.332425,7.366037,9.332019,7.367054,9.332831,0.0,"POLYGON ((7.36704 9.33202, 7.36705 9.33283, 7....",2,0
9,3,7.367552,9.332425,7.367043,9.332019,7.368060,9.332831,0.0,"POLYGON ((7.36805 9.33202, 7.36806 9.33283, 7....",2,0
12,4,7.368558,9.332425,7.368049,9.332019,7.369066,9.332831,0.0,"POLYGON ((7.36906 9.33202, 7.36907 9.33283, 7....",2,0
...,...,...,...,...,...,...,...,...,...,...,...
427326,142495,7.725878,8.894373,7.725370,8.893967,7.726386,8.894779,0.0,"POLYGON ((7.72638 8.89397, 7.72639 8.89478, 7....",2,0
427329,142496,7.726883,8.894373,7.726375,8.893967,7.727391,8.894779,0.0,"POLYGON ((7.72738 8.89397, 7.72739 8.89478, 7....",2,0
427332,142497,7.727888,8.894373,7.727380,8.893967,7.728396,8.894779,0.0,"POLYGON ((7.72839 8.89397, 7.7284 8.89478, 7.7...",2,0
427335,142498,7.728894,8.894373,7.728386,8.893967,7.729402,8.894779,0.0,"POLYGON ((7.72939 8.89397, 7.7294 8.89478, 7.7...",2,0


In [94]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_grid = results_grid.drop(columns=['Accessibility_standard','grid_id', 'geometry'])
results_grid.to_csv(model_outputs + 'model-output.csv', index=False)